In [1]:
!pip install -q transformers datasets peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.7 MB/s eta 0:00:00


In [2]:
from datasets import load_dataset

dataset = load_dataset("blended_skill_talk", split="train")
print(dataset[0])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/5.88M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/2.62M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/2.40M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4819 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1009 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/980 [00:00<?, ? examples/s]

{'personas': ["i've 2 kids.", 'i love flowers.'], 'additional_context': '', 'previous_utterance': ["I love live music, that's why I try to go to concerts", 'I do too. Wat do you like?'], 'context': 'empathetic_dialogues', 'free_messages': ['I like acting, I hope to be an actor, what about you?', 'No, but someday.', 'After I am done with school I plan to have a family.', 'I hope so, how old are your kids?', 'I would imagine. I am sure they a great kids.', 'I wish I had more time to do stuff like that. Medical school is exhausting. '], 'guided_messages': ['that is ok.  have any kids?', 'that is good. I have 2', 'that is great! you will be ready', '5 & 7.  they take up a lot of my time', 'luckily, they love flowers just as much as I do.  we spend a lot of time in the garden', 'sounds like it. have you gotten any acting jobs, though?'], 'suggestions': {'convai2': ["i love acting ! i'll be famous someday . what do you do ?", 'no no kids , might get some though . one day', 'that is great . i

In [3]:
example = dataset[0]
free_messages = example["free_messages"]
guided_messages = example["guided_messages"]

for user, bot in zip(free_messages, guided_messages):
    print("User:", user)
    print("Bot:", bot)
    print("---")

User: I like acting, I hope to be an actor, what about you?
Bot: that is ok.  have any kids?
---
User: No, but someday.
Bot: that is good. I have 2
---
User: After I am done with school I plan to have a family.
Bot: that is great! you will be ready
---
User: I hope so, how old are your kids?
Bot: 5 & 7.  they take up a lot of my time
---
User: I would imagine. I am sure they a great kids.
Bot: luckily, they love flowers just as much as I do.  we spend a lot of time in the garden
---
User: I wish I had more time to do stuff like that. Medical school is exhausting. 
Bot: sounds like it. have you gotten any acting jobs, though?
---


In [4]:
def format_example(example):
    pairs = []

    free_messages = example["free_messages"]
    guided_messages = example["guided_messages"]

    for user, bot in zip(free_messages, guided_messages):
        if user.strip() and bot.strip():
            pairs.append(f"User: {user}\nBot: {bot}")

    return {"text": pairs}

In [5]:
dataset = dataset.map(format_example)

Map:   0%|          | 0/4819 [00:00<?, ? examples/s]

In [6]:
dataset[0]['text']

['User: I like acting, I hope to be an actor, what about you?\nBot: that is ok.  have any kids?',
 'User: No, but someday.\nBot: that is good. I have 2',
 'User: After I am done with school I plan to have a family.\nBot: that is great! you will be ready',
 'User: I hope so, how old are your kids?\nBot: 5 & 7.  they take up a lot of my time',
 'User: I would imagine. I am sure they a great kids.\nBot: luckily, they love flowers just as much as I do.  we spend a lot of time in the garden',
 'User: I wish I had more time to do stuff like that. Medical school is exhausting. \nBot: sounds like it. have you gotten any acting jobs, though?']

In [7]:
from itertools import chain

all_texts = list(chain.from_iterable(dataset["text"]))

from datasets import Dataset
new_dataset = Dataset.from_dict({"text": all_texts})

In [9]:
new_dataset[0]['text']

'User: I like acting, I hope to be an actor, what about you?\nBot: that is ok.  have any kids?'

In [10]:
from transformers import AutoTokenizer

model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [11]:
def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    labels = tokens["input_ids"].copy()

    labels = [
        token if token != tokenizer.pad_token_id else -100
        for token in labels
    ]

    tokens["labels"] = labels

    return tokens

tokenized_dataset = new_dataset.map(tokenize)

Map:   0%|          | 0/27017 [00:00<?, ? examples/s]

In [12]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(model_name)

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [13]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["c_attn"],  # GPT2 attention layers
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [14]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=8,
    num_train_epochs=5,   # keep small for Colab
    logging_steps=50,
    save_steps=200,
    learning_rate=2e-4,
    fp16=True
)

In [15]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

In [16]:
trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
50,4.123070
100,3.702676
150,3.576161
200,3.542963
250,3.483556
300,3.410508
350,3.439533
400,3.410981
450,3.404022
500,3.455117


TrainOutput(global_step=16890, training_loss=3.2953208132846035, metrics={'train_runtime': 1149.4941, 'train_samples_per_second': 117.517, 'train_steps_per_second': 14.693, 'total_flos': 4427456808222720.0, 'train_loss': 3.2953208132846035, 'epoch': 5.0})

In [17]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): GPT2LMHeadModel(
      (transformer): GPT2Model(
        (wte): Embedding(50257, 768)
        (wpe): Embedding(1024, 768)
        (drop): Dropout(p=0.1, inplace=False)
        (h): ModuleList(
          (0-5): 6 x GPT2Block(
            (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (attn): GPT2Attention(
              (c_attn): lora.Linear(
                (base_layer): Conv1D(nf=2304, nx=768)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=768, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2304, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
             

In [20]:
def chat():
    print("🧠 Mental Health Chatbot (type 'exit' to stop)\n")

    while True:
        user_input = input("You: ")

        if user_input.lower() == "exit":
            print("Bot: Take care 💙")
            break

        prompt = f"User: {user_input}\nBot:"

        inputs = tokenizer(prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = model.generate(
              **inputs,
    max_length=100,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.2,     # 🔥 prevents repetition
    no_repeat_ngram_size=3      # 🔥 blocks repeated phrases
)

        response = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # extract only bot part
        reply = response.split("Bot:")[-1].strip()

        print("Bot:", reply)

In [21]:
chat()

🧠 Mental Health Chatbot (type 'exit' to stop)

You: i feel depressed


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Bot: That's true.  You can't just relax and focus on what is important to you, because it would be a mistake for me to stop doing so anyway! What about your pets? I'm very worried that their fur will go up in flames if they don�t get used to the fact that we are not being careful enough ! How do you like them all ? Do you have dogs or cats , even though my pet has some problems with him??
You: exit
Bot: Take care 💙
